# Feature Extraction: Intraday Session Features

This notebook implements **Features 14-15** (After-Hours / Market-Hours) and **Features 40-48** (Intraday Sessions + Volatility), plus **weekend** and **holiday** sentiment/volume.

**Session Classification Logic:**
Using columns from `data_cleaning.ipynb` already present in the merged data:
- `session` (int64, HHMM format): last half-hour mark before tweet time (e.g., 930 = 09:30, 1600 = 16:00)
- `business_day` (bool): whether the tweet was posted on a trading day
- `is_weekend` (bool): whether the tweet was posted on a weekend
- `is_holiday` (bool): whether the tweet was posted on a market holiday

**Business Day Sessions (Features 40-47):**

| # | Session | Time Range | Session HHMM |
|---|---------|------------|-------------|
| 40 | Midnight to Morning | 00:00 - 09:00 | < 900 |
| 41 | Pre-Market | 09:00 - 09:30 | 900 |
| 42 | Market Open | 09:30 - 10:00 | 930 |
| 43 | Late Morning | 10:00 - 12:00 | 1000 - 1130 |
| 44 | Mid-Day | 12:00 - 13:00 | 1200 - 1230 |
| 45 | Early Afternoon | 13:00 - 15:30 | 1300 - 1500 |
| 46 | Market Close | 15:30 - 16:00 | 1530 |
| 47 | Post-Market | 16:00 - 23:59 | >= 1600 |

**Non-Business Day Variables:**
- Weekend: Volume & Sentiment for tweets posted on Saturdays/Sundays
- Holiday: Volume & Sentiment for tweets posted on market holidays (non-trading, non-weekend)

**Aggregated (Features 14-15):**
- After-Hours (Feature 14): midnight_to_morning + pre_market + post_market (business days only)
- Market-Hours (Feature 15): market_open + late_morning + midday + early_afternoon + market_close

**Feature 48:** Intraday Sentiment Volatility (std dev across the 8 business day sessions)

**Output:** `features_04_intraday_sessions.pkl`

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
INPUT_FOLDER = DATA_DIR / "merged_with_crsp_mlcrowd"
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_FOLDER / "features_04_intraday_sessions.pkl"

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output file: {OUTPUT_FILE}")

# =============================================================================
# SESSION DEFINITIONS
# =============================================================================
BUSINESS_DAY_SESSIONS = [
    'midnight_to_morning',  # 00:00 - 09:00  (session < 900)
    'pre_market',           # 09:00 - 09:30  (session = 900)
    'market_open',          # 09:30 - 10:00  (session = 930)
    'late_morning',         # 10:00 - 12:00  (session 1000-1130)
    'midday',               # 12:00 - 13:00  (session 1200-1230)
    'early_afternoon',      # 13:00 - 15:30  (session 1300-1500)
    'market_close',         # 15:30 - 16:00  (session = 1530)
    'post_market',          # 16:00 - 23:59  (session >= 1600)
]

AFTER_HOURS_SESSIONS = {'midnight_to_morning', 'pre_market', 'post_market'}
MARKET_HOURS_SESSIONS = {'market_open', 'late_morning', 'midday', 'early_afternoon', 'market_close'}

ALL_GROUPS = BUSINESS_DAY_SESSIONS + ['weekend', 'holiday']

## 1. Load Sample Data for Development

In [ ]:
files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith('.csv')])
print(f"Available files: {len(files)}")
print(f"Files: {files}")

# Load most recent year for development
sample_file = INPUT_FOLDER / files[-1]
print(f"\nLoading sample: {sample_file.name}")

df_sample = pd.read_csv(sample_file)
df_sample['date'] = pd.to_datetime(df_sample['date'])

print(f"Shape: {df_sample.shape}")
print(f"Date range: {df_sample['date'].min()} to {df_sample['date'].max()}")
print(f"Unique symbols: {df_sample['symbol'].nunique():,}")

# Verify required columns exist
required_cols = ['session', 'business_day', 'is_weekend', 'is_holiday', 'sentiment', 'message_id']
for col in required_cols:
    assert col in df_sample.columns, f"Missing required column: {col}"
print(f"\nAll required columns present.")

# Inspect key columns
print(f"\nColumn types:")
print(f"  session:      {df_sample['session'].dtype} (sample: {df_sample['session'].iloc[:5].tolist()})")
print(f"  business_day: {df_sample['business_day'].dtype} (sample: {df_sample['business_day'].iloc[:5].tolist()})")
print(f"  is_weekend:   {df_sample['is_weekend'].dtype} (sample: {df_sample['is_weekend'].iloc[:5].tolist()})")
print(f"  is_holiday:   {df_sample['is_holiday'].dtype} (sample: {df_sample['is_holiday'].iloc[:5].tolist()})")

# Day type distribution
n_bd = df_sample['business_day'].sum()
n_wk = df_sample['is_weekend'].sum()
n_hol = df_sample['is_holiday'].sum()
print(f"\nDay type distribution:")
print(f"  Business day: {n_bd:,} tweets ({n_bd/len(df_sample)*100:.1f}%)")
print(f"  Weekend:      {n_wk:,} tweets ({n_wk/len(df_sample)*100:.1f}%)")
print(f"  Holiday:      {n_hol:,} tweets ({n_hol/len(df_sample)*100:.1f}%)")

# Session HHMM distribution
print(f"\nSession (HHMM) value counts (top 15):")
print(df_sample['session'].value_counts().head(15))

## 2. Define Session Group Mapping

Maps each tweet to one of 10 groups using the `session` (int64 HHMM), `business_day`, `is_weekend`, and `is_holiday` columns directly from the data.

In [ ]:
def map_to_group(df):
    """
    Map each tweet to a session group using business_day, is_weekend, is_holiday, session.
    
    Non-business days are classified first (weekend/holiday take priority).
    Business day tweets are classified into 8 intraday sessions based on the
    session column (int64 HHMM format from data_cleaning.ipynb).
    """
    s = df['session']
    bd = df['business_day']
    
    conditions = [
        # Non-business days (check first)
        df['is_weekend'],
        df['is_holiday'],
        # Business day: 8 intraday sessions
        bd & (s < 900),                       # midnight_to_morning: 00:00 - 09:00
        bd & (s >= 900) & (s < 930),          # pre_market:          09:00 - 09:30
        bd & (s >= 930) & (s < 1000),         # market_open:         09:30 - 10:00
        bd & (s >= 1000) & (s < 1200),        # late_morning:        10:00 - 12:00
        bd & (s >= 1200) & (s < 1300),        # midday:              12:00 - 13:00
        bd & (s >= 1300) & (s < 1530),        # early_afternoon:     13:00 - 15:30
        bd & (s >= 1530) & (s < 1600),        # market_close:        15:30 - 16:00
        bd & (s >= 1600),                     # post_market:         16:00 - 23:59
    ]
    choices = [
        'weekend', 'holiday',
        'midnight_to_morning', 'pre_market', 'market_open', 'late_morning',
        'midday', 'early_afternoon', 'market_close', 'post_market',
    ]
    return np.select(conditions, choices, default='unknown')

## 3. Test Session Classification on Sample

In [ ]:
df_test = df_sample.copy()
df_test['group'] = map_to_group(df_test)

print("Group classification:")
print(df_test['group'].value_counts())

print(f"\nUnknown groups: {(df_test['group'] == 'unknown').sum()}")

# Cross-tab: session HHMM vs group (for business days)
bd_mask = df_test['business_day']
print(f"\nCross-tab of session HHMM vs group (business days only, top sessions):")
ct = pd.crosstab(df_test.loc[bd_mask, 'session'], df_test.loc[bd_mask, 'group'])
display(ct.head(20))

# Show a few examples
print(f"\nSample rows with classification:")
display(df_test[['created_at', 'time', 'session', 'business_day', 'is_weekend', 'is_holiday', 'group', 'date', 'symbol']].head(15))

del df_test

## 4. Define Feature Calculation Function

In [ ]:
def calculate_intraday_features(df):
    """
    Calculate intraday session features for each stock-day.
    
    Uses existing columns: session (int64 HHMM), business_day, is_weekend, is_holiday.
    
    Returns a DataFrame with one row per (symbol, date) containing:
        - 8 business day sessions: {session}_volume, {session}_sentiment
        - weekend_volume, weekend_sentiment
        - holiday_volume, holiday_sentiment
        - after_hours_volume, after_hours_sentiment      (Feature 14)
        - market_hours_volume, market_hours_sentiment    (Feature 15)
        - intraday_sentiment_volatility                  (Feature 48)
    """
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    # --- Step 1: Classify into groups ---
    df['group'] = map_to_group(df)
    
    # --- Step 2: Sentiment indicators ---
    df['is_bullish'] = (df['sentiment'] == 'Bullish').astype(int)
    df['is_bearish'] = (df['sentiment'] == 'Bearish').astype(int)
    
    # --- Step 3: Aggregate by (symbol, date, group) ---
    group_agg = df.groupby(['symbol', 'date', 'group']).agg(
        volume=('message_id', 'count'),
        n_bullish=('is_bullish', 'sum'),
        n_bearish=('is_bearish', 'sum')
    ).reset_index()
    group_agg['total_labeled'] = group_agg['n_bullish'] + group_agg['n_bearish']
    group_agg['net_sentiment'] = np.where(
        group_agg['total_labeled'] > 0,
        (group_agg['n_bullish'] - group_agg['n_bearish']) / group_agg['total_labeled'],
        0.0
    )
    
    # --- Step 4: Pivot to (symbol, date) rows ---
    vol_pivot = group_agg.pivot_table(
        index=['symbol', 'date'], columns='group', values='volume', fill_value=0
    )
    vol_pivot.columns = [f'{c}_volume' for c in vol_pivot.columns]
    
    sent_pivot = group_agg.pivot_table(
        index=['symbol', 'date'], columns='group', values='net_sentiment', fill_value=0.0
    )
    sent_pivot.columns = [f'{c}_sentiment' for c in sent_pivot.columns]
    
    features = vol_pivot.join(sent_pivot).reset_index()
    
    # Ensure all 10 groups have columns
    for g in ALL_GROUPS:
        if f'{g}_volume' not in features.columns:
            features[f'{g}_volume'] = 0
        if f'{g}_sentiment' not in features.columns:
            features[f'{g}_sentiment'] = 0.0
    
    vol_cols = [f'{g}_volume' for g in ALL_GROUPS]
    sent_cols = [f'{g}_sentiment' for g in ALL_GROUPS]
    features[vol_cols] = features[vol_cols].fillna(0).astype(int)
    features[sent_cols] = features[sent_cols].fillna(0.0)
    
    # --- Step 5: After-Hours / Market-Hours aggregation (Features 14-15) ---
    # Computed from raw bullish/bearish counts (not averaged sentiments)
    def aggregate_groups(agg_df, group_set, prefix):
        subset = agg_df[agg_df['group'].isin(group_set)]
        if len(subset) == 0:
            return pd.DataFrame(columns=['symbol', 'date', f'{prefix}_volume', f'{prefix}_sentiment'])
        result = subset.groupby(['symbol', 'date']).agg(
            volume=('volume', 'sum'),
            n_bullish=('n_bullish', 'sum'),
            n_bearish=('n_bearish', 'sum')
        ).reset_index()
        total_labeled = result['n_bullish'] + result['n_bearish']
        result[f'{prefix}_sentiment'] = np.where(
            total_labeled > 0,
            (result['n_bullish'] - result['n_bearish']) / total_labeled,
            0.0
        )
        result = result.rename(columns={'volume': f'{prefix}_volume'})
        return result[['symbol', 'date', f'{prefix}_volume', f'{prefix}_sentiment']]
    
    ah_feats = aggregate_groups(group_agg, AFTER_HOURS_SESSIONS, 'after_hours')
    mh_feats = aggregate_groups(group_agg, MARKET_HOURS_SESSIONS, 'market_hours')
    
    features = features.merge(ah_feats, on=['symbol', 'date'], how='left')
    features = features.merge(mh_feats, on=['symbol', 'date'], how='left')
    
    for col in ['after_hours_volume', 'market_hours_volume']:
        features[col] = features[col].fillna(0).astype(int)
    for col in ['after_hours_sentiment', 'market_hours_sentiment']:
        features[col] = features[col].fillna(0.0)
    
    # --- Step 6: Intraday Sentiment Volatility (Feature 48) ---
    bd_sent_cols = [f'{s}_sentiment' for s in BUSINESS_DAY_SESSIONS]
    features['intraday_sentiment_volatility'] = features[bd_sent_cols].std(axis=1)
    
    # --- Step 7: Order columns logically ---
    ordered_cols = ['symbol', 'date',
                    'after_hours_volume', 'after_hours_sentiment',
                    'market_hours_volume', 'market_hours_sentiment']
    for s in BUSINESS_DAY_SESSIONS:
        ordered_cols.extend([f'{s}_volume', f'{s}_sentiment'])
    ordered_cols.extend(['weekend_volume', 'weekend_sentiment',
                         'holiday_volume', 'holiday_sentiment',
                         'intraday_sentiment_volatility'])
    features = features[[c for c in ordered_cols if c in features.columns]]
    
    return features

## 5. Test on Sample Data

In [ ]:
features_sample = calculate_intraday_features(df_sample)

print(f"Features shape: {features_sample.shape}")
print(f"\nColumns ({len(features_sample.columns)}):")
for col in features_sample.columns:
    print(f"  {col}: {features_sample[col].dtype}")

print(f"\nSample features (first 10 rows):")
display(features_sample.head(10))

print(f"\nSummary statistics:")
display(features_sample.describe())

## 6. Visualize & Validate Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Average volume by session (business day sessions)
bd_vol_cols = [f'{s}_volume' for s in BUSINESS_DAY_SESSIONS]
bd_labels = ['Midnight-\n9AM', 'Pre-Mkt', 'Mkt Open', 'Late AM',
             'Midday', 'Early PM', 'Mkt Close', 'Post-Mkt']
mean_vols = [features_sample[c].mean() for c in bd_vol_cols]
axes[0, 0].bar(bd_labels, mean_vols, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Average Volume by Business Day Session')
axes[0, 0].set_ylabel('Mean Tweet Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Average sentiment by session (business day sessions)
bd_sent_cols = [f'{s}_sentiment' for s in BUSINESS_DAY_SESSIONS]
mean_sents = [features_sample[c].mean() for c in bd_sent_cols]
colors = ['green' if s >= 0 else 'red' for s in mean_sents]
axes[0, 1].bar(bd_labels, mean_sents, color=colors, edgecolor='black')
axes[0, 1].set_title('Average Net Sentiment by Business Day Session')
axes[0, 1].set_ylabel('Mean Net Sentiment')
axes[0, 1].axhline(0, color='black', linewidth=0.5)
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Total volume by day type
vol_ah = features_sample['after_hours_volume'].sum()
vol_mh = features_sample['market_hours_volume'].sum()
vol_wk = features_sample['weekend_volume'].sum()
vol_hol = features_sample['holiday_volume'].sum()
day_labels = ['After-Hours\n(Biz Day)', 'Market-Hours\n(Biz Day)', 'Weekend', 'Holiday']
day_vols = [vol_ah, vol_mh, vol_wk, vol_hol]
day_colors = ['#ff7f0e', '#1f77b4', '#2ca02c', '#d62728']
bars = axes[1, 0].bar(day_labels, day_vols, color=day_colors, edgecolor='black')
axes[1, 0].set_title('Total Volume by Day Type')
axes[1, 0].set_ylabel('Total Tweet Count')
for bar, v in zip(bars, day_vols):
    if v > 0:
        axes[1, 0].text(bar.get_x() + bar.get_width()/2, v + v*0.01,
                        f'{v:,}', ha='center', fontweight='bold', fontsize=9)

# 4. Intraday sentiment volatility distribution
features_sample['intraday_sentiment_volatility'].dropna().hist(
    bins=50, ax=axes[1, 1], edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Distribution of Intraday Sentiment Volatility')
axes[1, 1].set_xlabel('Std Dev of Sentiment Across 8 Business Day Sessions')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 7. Validation Checks

In [ ]:
print("Validation Checks")
print("=" * 60)

# Check 1: Business day session volumes = after_hours + market_hours
bd_vol_cols = [f'{s}_volume' for s in BUSINESS_DAY_SESSIONS]
total_bd = features_sample[bd_vol_cols].sum(axis=1)
total_ah_mh = features_sample['after_hours_volume'] + features_sample['market_hours_volume']
print(f"\n1. Business day sessions vs After-Hours + Market-Hours:")
mismatch = (total_bd != total_ah_mh).sum()
print(f"   Mismatches: {mismatch} / {len(features_sample)}")

# Check 2: Sentiment bounds
all_sent_cols = [f'{g}_sentiment' for g in ALL_GROUPS] + ['after_hours_sentiment', 'market_hours_sentiment']
all_sents = features_sample[all_sent_cols]
print(f"\n2. Sentiment bounds [-1, 1]:")
print(f"   Min: {all_sents.min().min():.4f}")
print(f"   Max: {all_sents.max().max():.4f}")

# Check 3: No NaN values
print(f"\n3. NaN check (should be 0 for all feature columns):")
feature_cols = [c for c in features_sample.columns if c not in ['symbol', 'date']]
nan_total = features_sample[feature_cols].isnull().sum().sum()
print(f"   Total NaN across all feature columns: {nan_total}")

# Check 4: Session coverage
print(f"\n4. Group coverage (stock-days with activity):")
for g in ALL_GROUPS:
    n_nonzero = (features_sample[f'{g}_volume'] > 0).sum()
    pct = n_nonzero / len(features_sample) * 100
    print(f"   {g:25s}: {n_nonzero:6,} ({pct:5.1f}%)")

# Check 5: After-hours composition
print(f"\n5. After-Hours = midnight_to_morning + pre_market + post_market:")
ah_from_sessions = (features_sample['midnight_to_morning_volume'] +
                    features_sample['pre_market_volume'] +
                    features_sample['post_market_volume'])
ah_match = (ah_from_sessions == features_sample['after_hours_volume']).all()
print(f"   Volume match: {ah_match}")

# Check 6: Market-hours composition
print(f"\n6. Market-Hours = market_open + late_morning + midday + early_afternoon + market_close:")
mh_from_sessions = (features_sample['market_open_volume'] +
                    features_sample['late_morning_volume'] +
                    features_sample['midday_volume'] +
                    features_sample['early_afternoon_volume'] +
                    features_sample['market_close_volume'])
mh_match = (mh_from_sessions == features_sample['market_hours_volume']).all()
print(f"   Volume match: {mh_match}")

## 8. Process All Years

In [ ]:
all_features = []
processing_stats = []

print(f"{'=' * 60}")
print(f"Processing all years...")
print(f"{'=' * 60}\n")

for file in tqdm(files, desc="Processing years"):
    try:
        year = file.split('_')[-1].replace('.csv', '')
        file_path = INPUT_FOLDER / file
        
        df_year = pd.read_csv(file_path)
        features_year = calculate_intraday_features(df_year)
        all_features.append(features_year)
        
        processing_stats.append({
            'year': year,
            'input_rows': len(df_year),
            'feature_rows': len(features_year),
            'unique_symbols': features_year['symbol'].nunique(),
            'unique_dates': features_year['date'].nunique()
        })
        
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")
        import traceback
        traceback.print_exc()

features_all = pd.concat(all_features, ignore_index=True)

print(f"\n{'=' * 60}")
print(f"Processing Complete!")
print(f"{'=' * 60}")
print(f"\nTotal rows: {len(features_all):,}")
print(f"Unique symbols: {features_all['symbol'].nunique():,}")
print(f"Date range: {features_all['date'].min()} to {features_all['date'].max()}")

df_stats = pd.DataFrame(processing_stats)
print(f"\nProcessing Statistics by Year:")
display(df_stats)

## 9. Final Data Inspection

In [ ]:
print(f"Final Features Dataset:")
print(f"{'=' * 60}")
print(f"Shape: {features_all.shape}")
print(f"\nColumns: {list(features_all.columns)}")
print(f"\nData types:")
print(features_all.dtypes)
print(f"\nMemory usage: {features_all.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nNull values:")
print(features_all.isnull().sum())
print(f"\nSummary statistics:")
display(features_all.describe())

## 10. Save to Pickle

In [ ]:
print(f"Saving features to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)
print(f"Saved successfully!")

# Verify save
print(f"\nVerifying saved file...")
features_loaded = pd.read_pickle(OUTPUT_FILE)
print(f"File readable: {True}")
print(f"Shape matches: {features_loaded.shape == features_all.shape}")
print(f"Columns match: {list(features_loaded.columns) == list(features_all.columns)}")

file_size_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"\nFile size: {file_size_mb:.2f} MB")

## Summary

**Features Implemented (25 columns):**

| Feature # | Name | Day Type | Columns |
|-----------|------|----------|--------|
| 14 | After-Hours Activity | Business Day | `after_hours_volume`, `after_hours_sentiment` |
| 15 | Market-Hours Activity | Business Day | `market_hours_volume`, `market_hours_sentiment` |
| 40 | Midnight to Morning (00:00-09:00) | Business Day | `midnight_to_morning_volume`, `midnight_to_morning_sentiment` |
| 41 | Pre-Market (09:00-09:30) | Business Day | `pre_market_volume`, `pre_market_sentiment` |
| 42 | Market Open (09:30-10:00) | Business Day | `market_open_volume`, `market_open_sentiment` |
| 43 | Late Morning (10:00-12:00) | Business Day | `late_morning_volume`, `late_morning_sentiment` |
| 44 | Mid-Day (12:00-13:00) | Business Day | `midday_volume`, `midday_sentiment` |
| 45 | Early Afternoon (13:00-15:30) | Business Day | `early_afternoon_volume`, `early_afternoon_sentiment` |
| 46 | Market Close (15:30-16:00) | Business Day | `market_close_volume`, `market_close_sentiment` |
| 47 | Post-Market (16:00-23:59) | Business Day | `post_market_volume`, `post_market_sentiment` |
| - | Weekend Activity | Weekend | `weekend_volume`, `weekend_sentiment` |
| - | Holiday Activity | Holiday | `holiday_volume`, `holiday_sentiment` |
| 48 | Intraday Sentiment Volatility | Business Day | `intraday_sentiment_volatility` |

**Output:** `features_04_intraday_sessions.pkl`

**Key Design Decisions:**
- `session` (int64 HHMM), `business_day`, `is_weekend`, `is_holiday` used directly from the data (computed in `data_cleaning.ipynb`)
- Non-business days (weekend/holiday) classified first, then business days split into 8 intraday sessions
- After-Hours = midnight_to_morning + pre_market + post_market (business days only)
- Market-Hours = market_open + late_morning + midday + early_afternoon + market_close
- After-Hours/Market-Hours sentiment computed from raw bullish/bearish counts (not averaged across sub-sessions)
- Sentiment = 0 (neutral) when no labeled tweets in a session
- Intraday volatility = std dev across 8 business day session sentiments